In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load final scored dataset
df = pd.read_csv('data/processed/nike_final_scored.csv')
print(f"Loaded {len(df)} tweets")
print(f"Columns: {list(df.columns)}")
print(f"\nVADER distribution:\n{df['vader_label'].value_counts()}")
print(f"\nBERT distribution:\n{df['bert_label'].value_counts()}")
print(f"\nTop topics:\n{df['topic_label'].value_counts().head(5)}")

Loaded 9941 tweets
Columns: ['sentiment', 'id', 'date', 'query', 'user', 'text', 'text_length', 'word_count', 'clean_text', 'clean_word_count', 'vader_pos', 'vader_neg', 'vader_neu', 'vader_compound', 'vader_label', 'bert_label', 'bert_score', 'topic', 'topic_label']

VADER distribution:
vader_label
positive    4785
neutral     2759
negative    2397
Name: count, dtype: int64

BERT distribution:
bert_label
neutral     4706
positive    2867
negative    2368
Name: count, dtype: int64

Top topics:
topic_label
Other                     6260
Food & Dining              563
Sleep & Fatigue            444
General Conversation       420
Twitter & Social Media     347
Name: count, dtype: int64


In [3]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

app = Dash(__name__)

# ── Colour scheme ──────────────────────────────────────────
COLORS = {
    'positive': '#1D9E75',
    'neutral':  '#EF9F27',
    'negative': '#E24B4A',
    'background': '#F8F9FA',
    'card': '#FFFFFF',
    'header': '#1A3C5E'
}

# ── Pre-compute data ───────────────────────────────────────
vader_counts = df['vader_label'].value_counts().reset_index()
vader_counts.columns = ['sentiment', 'count']

bert_counts = df['bert_label'].value_counts().reset_index()
bert_counts.columns = ['sentiment', 'count']

topic_counts = df['topic_label'].value_counts().reset_index()
topic_counts = topic_counts[topic_counts['topic_label'] != 'Other'].head(8)
topic_counts.columns = ['topic', 'count']

model_comparison = pd.DataFrame({
    'Model': ['VADER', 'VADER', 'VADER',
              'BERT',  'BERT',  'BERT'],
    'Sentiment': ['positive', 'neutral', 'negative',
                  'positive', 'neutral', 'negative'],
    'Percentage': [48.1, 27.8, 24.1,
                   28.8, 47.3, 23.8]
})

# ── Layout ─────────────────────────────────────────────────
app.layout = html.Div(style={'backgroundColor': COLORS['background'],
                              'fontFamily': 'Arial, sans-serif',
                              'minHeight': '100vh'}, children=[

    # Header
    html.Div(style={
        'backgroundColor': COLORS['header'],
        'padding': '20px 40px',
        'marginBottom': '20px'
    }, children=[
        html.H1("BrandLens.AI — Social Media Analytics Dashboard",
                style={'color': 'white', 'margin': 0, 'fontSize': '24px'}),
        html.P("NLP-Powered Brand Intelligence Framework | Nike Case Study",
               style={'color': '#B0C4DE', 'margin': '5px 0 0', 'fontSize': '13px'})
    ]),

    # KPI Cards Row
    html.Div(style={
        'display': 'flex', 'gap': '15px',
        'padding': '0 40px', 'marginBottom': '20px'
    }, children=[
        # Card 1
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'flex': 1, 'borderLeft': f"4px solid {COLORS['header']}"
        }, children=[
            html.P("Total Tweets Analysed",
                   style={'color': '#888', 'fontSize': '12px', 'margin': 0}),
            html.H2("9,941", style={'margin': '5px 0 0',
                                    'color': COLORS['header'], 'fontSize': '28px'})
        ]),
        # Card 2
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'flex': 1, 'borderLeft': f"4px solid {COLORS['positive']}"
        }, children=[
            html.P("Positive Sentiment (BERT)",
                   style={'color': '#888', 'fontSize': '12px', 'margin': 0}),
            html.H2("28.8%", style={'margin': '5px 0 0',
                                    'color': COLORS['positive'], 'fontSize': '28px'})
        ]),
        # Card 3
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'flex': 1, 'borderLeft': f"4px solid {COLORS['negative']}"
        }, children=[
            html.P("Negative Sentiment (BERT)",
                   style={'color': '#888', 'fontSize': '12px', 'margin': 0}),
            html.H2("23.8%", style={'margin': '5px 0 0',
                                    'color': COLORS['negative'], 'fontSize': '28px'})
        ]),
        # Card 4
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'flex': 1, 'borderLeft': f"4px solid {COLORS['neutral']}"
        }, children=[
            html.P("BERT Model Accuracy",
                   style={'color': '#888', 'fontSize': '12px', 'margin': 0}),
            html.H2("80%", style={'margin': '5px 0 0',
                                  'color': COLORS['neutral'], 'fontSize': '28px'})
        ]),
        # Card 5
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'flex': 1, 'borderLeft': f"4px solid #534AB7"
        }, children=[
            html.P("Topics Identified",
                   style={'color': '#888', 'fontSize': '12px', 'margin': 0}),
            html.H2("28", style={'margin': '5px 0 0',
                                 'color': '#534AB7', 'fontSize': '28px'})
        ]),
    ]),

    # Filter Row
    html.Div(style={'padding': '0 40px', 'marginBottom': '20px'}, children=[
        html.Div(style={
            'backgroundColor': COLORS['card'],
            'borderRadius': '8px', 'padding': '15px 20px',
            'display': 'flex', 'alignItems': 'center', 'gap': '20px'
        }, children=[
            html.Label("Filter by Model:",
                       style={'fontWeight': 'bold', 'color': COLORS['header'],
                              'fontSize': '13px'}),
            dcc.RadioItems(
                id='model-filter',
                options=[
                    {'label': ' VADER', 'value': 'vader'},
                    {'label': ' BERT',  'value': 'bert'}
                ],
                value='bert',
                inline=True,
                style={'fontSize': '13px'}
            ),
            html.Label("Filter by Sentiment:",
                       style={'fontWeight': 'bold', 'color': COLORS['header'],
                              'fontSize': '13px', 'marginLeft': '20px'}),
            dcc.Dropdown(
                id='sentiment-filter',
                options=[
                    {'label': 'All',      'value': 'all'},
                    {'label': 'Positive', 'value': 'positive'},
                    {'label': 'Neutral',  'value': 'neutral'},
                    {'label': 'Negative', 'value': 'negative'}
                ],
                value='all',
                style={'width': '150px', 'fontSize': '13px'}
            )
        ])
    ]),

    # Charts Row 1
    html.Div(style={
        'display': 'flex', 'gap': '15px',
        'padding': '0 40px', 'marginBottom': '15px'
    }, children=[
        # Sentiment Pie
        html.Div(style={
            'backgroundColor': COLORS['card'], 'borderRadius': '8px',
            'padding': '15px', 'flex': 1
        }, children=[
            html.H4("Sentiment Distribution",
                    style={'color': COLORS['header'], 'margin': '0 0 10px',
                           'fontSize': '14px'}),
            dcc.Graph(id='sentiment-pie', style={'height': '280px'})
        ]),
        # Model Comparison
        html.Div(style={
            'backgroundColor': COLORS['card'], 'borderRadius': '8px',
            'padding': '15px', 'flex': 1
        }, children=[
            html.H4("VADER vs BERT Comparison",
                    style={'color': COLORS['header'], 'margin': '0 0 10px',
                           'fontSize': '14px'}),
            dcc.Graph(id='model-comparison', style={'height': '280px'})
        ]),
        # Topic Distribution
        html.Div(style={
            'backgroundColor': COLORS['card'], 'borderRadius': '8px',
            'padding': '15px', 'flex': 1
        }, children=[
            html.H4("Top Topic Clusters",
                    style={'color': COLORS['header'], 'margin': '0 0 10px',
                           'fontSize': '14px'}),
            dcc.Graph(id='topic-bar', style={'height': '280px'})
        ]),
    ]),

    # Charts Row 2
    html.Div(style={
        'display': 'flex', 'gap': '15px',
        'padding': '0 40px', 'marginBottom': '15px'
    }, children=[
        # Compound Score Distribution
        html.Div(style={
            'backgroundColor': COLORS['card'], 'borderRadius': '8px',
            'padding': '15px', 'flex': 2
        }, children=[
            html.H4("VADER Compound Score Distribution",
                    style={'color': COLORS['header'], 'margin': '0 0 10px',
                           'fontSize': '14px'}),
            dcc.Graph(id='compound-hist', style={'height': '250px'})
        ]),
        # BERT Confidence
        html.Div(style={
            'backgroundColor': COLORS['card'], 'borderRadius': '8px',
            'padding': '15px', 'flex': 1
        }, children=[
            html.H4("BERT Confidence Score",
                    style={'color': COLORS['header'], 'margin': '0 0 10px',
                           'fontSize': '14px'}),
            dcc.Graph(id='bert-confidence', style={'height': '250px'})
        ]),
    ]),

    # Footer
    html.Div(style={
        'textAlign': 'center', 'padding': '20px',
        'color': '#888', 'fontSize': '12px'
    }, children=[
        html.P("BrandLens.AI — MBA Data Science Capstone Project | "
               "Powered by VADER, BERT & BERTopic")
    ])
])

# ── Callbacks ──────────────────────────────────────────────
@app.callback(
    Output('sentiment-pie', 'figure'),
    Output('model-comparison', 'figure'),
    Output('topic-bar', 'figure'),
    Output('compound-hist', 'figure'),
    Output('bert-confidence', 'figure'),
    Input('model-filter', 'value'),
    Input('sentiment-filter', 'value')
)
def update_charts(model, sentiment_filter):
    # Filter data
    label_col = 'vader_label' if model == 'vader' else 'bert_label'
    dff = df.copy()
    if sentiment_filter != 'all':
        dff = dff[dff[label_col] == sentiment_filter]

    # 1. Sentiment Pie
    counts = dff[label_col].value_counts().reset_index()
    counts.columns = ['sentiment', 'count']
    pie_fig = px.pie(counts, names='sentiment', values='count',
                     color='sentiment',
                     color_discrete_map=COLORS,
                     hole=0.4)
    pie_fig.update_layout(margin=dict(t=10, b=10, l=10, r=10),
                          showlegend=True,
                          paper_bgcolor='white')
    pie_fig.update_traces(textposition='inside', textinfo='percent+label')

    # 2. Model Comparison Bar
    comp_fig = px.bar(model_comparison, x='Sentiment', y='Percentage',
                      color='Model', barmode='group',
                      color_discrete_map={'VADER': '#2E75B6', 'BERT': '#1D9E75'})
    comp_fig.update_layout(margin=dict(t=10, b=10, l=10, r=10),
                           paper_bgcolor='white',
                           plot_bgcolor='white',
                           yaxis_title='Percentage (%)')

    # 3. Topic Bar
    topic_fig = px.bar(topic_counts, x='count', y='topic',
                       orientation='h',
                       color_discrete_sequence=['#2E75B6'])
    topic_fig.update_layout(margin=dict(t=10, b=10, l=10, r=10),
                             paper_bgcolor='white',
                             plot_bgcolor='white',
                             yaxis={'categoryorder': 'total ascending'},
                             xaxis_title='Tweet Count',
                             yaxis_title='')

    # 4. Compound Score Histogram
    hist_fig = px.histogram(dff, x='vader_compound', nbins=50,
                             color_discrete_sequence=['#2E75B6'])
    hist_fig.add_vline(x=0.05, line_dash='dash', line_color='green',
                       annotation_text='Positive threshold')
    hist_fig.add_vline(x=-0.05, line_dash='dash', line_color='red',
                       annotation_text='Negative threshold')
    hist_fig.update_layout(margin=dict(t=10, b=10, l=10, r=10),
                            paper_bgcolor='white',
                            plot_bgcolor='white',
                            xaxis_title='Compound Score',
                            yaxis_title='Count')

    # 5. BERT Confidence
    conf_fig = px.histogram(dff, x='bert_score', nbins=30,
                             color_discrete_sequence=['#1D9E75'])
    conf_fig.update_layout(margin=dict(t=10, b=10, l=10, r=10),
                            paper_bgcolor='white',
                            plot_bgcolor='white',
                            xaxis_title='Confidence Score',
                            yaxis_title='Count')

    return pie_fig, comp_fig, topic_fig, hist_fig, conf_fig

# ── Run ────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8050)